In [1]:
import duckdb

Atur File pizza_types.csv

In [53]:
import pandas as pd

try:
    df_pizza_types = pd.read_csv('pizza_types.csv', encoding='utf-8')
except UnicodeDecodeError:
    try:
        df_pizza_types = pd.read_csv('pizza_types.csv', encoding='cp1252')
    except UnicodeDecodeError:
        df_pizza_types = pd.read_csv('pizza_types.csv', encoding='latin-1')

df_pizza_types.to_csv(
    'pizza_types_utf8.csv',
    index=False,
    encoding='utf-8'
)

# Buat tabel DuckDB
duckdb.sql("""
    CREATE OR REPLACE TABLE pizza_types AS
    SELECT * FROM read_csv_auto('pizza_types_utf8.csv')
""")

print("Tabel pizza_types berhasil dimuat!")
duckdb.sql("SELECT * FROM pizza_types LIMIT 5").show()

Tabel pizza_types berhasil dimuat!
┌───────────────┬──────────────────────────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────┐
│ pizza_type_id │             name             │ category │                                         ingredients                                          │
│    varchar    │           varchar            │ varchar  │                                           varchar                                            │
├───────────────┼──────────────────────────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────┤
│ bbq_ckn       │ The Barbecue Chicken Pizza   │ Chicken  │ Barbecued Chicken, Red Peppers, Green Peppers, Tomatoes, Red Onions, Barbecue Sauce          │
│ cali_ckn      │ The California Chicken Pizza │ Chicken  │ Chicken, Artichoke, Spinach, Garlic, Jalapeno Peppers, Fontina Cheese, Gouda Cheese          │
│ ckn_alfredo   │ The Chicken Alfre

Muat Seluruh File

In [54]:
# Load orders.csv
duckdb.sql("""
    CREATE OR REPLACE TABLE orders AS
    SELECT * FROM read_csv_auto('orders.csv')
""")

# Load order_details.csv
duckdb.sql("""
    CREATE OR REPLACE TABLE order_details AS
    SELECT * FROM read_csv_auto('order_details.csv')
""")

# Load pizzas.csv
duckdb.sql("""
    CREATE OR REPLACE TABLE pizzas AS
    SELECT * FROM read_csv_auto('pizzas.csv')
""")

print("Semua tabel lain berhasil dimuat ke DuckDB!")

Semua tabel lain berhasil dimuat ke DuckDB!


Buat Master File dengan Join

In [55]:
sql_create_master = """
CREATE OR REPLACE TABLE orders_full AS
SELECT
  od.order_details_id,
  od.order_id,
  o.date AS order_date,
  o.time AS order_time,

  MONTHNAME(CAST(o.date AS DATE)) AS month_name,
  DAYNAME(CAST(o.date AS DATE)) AS day_name,
  EXTRACT(HOUR FROM CAST(o.time AS TIME)) AS hour,

  od.pizza_id,
  od.quantity,
  p.pizza_type_id,
  pt.name AS pizza_type,
  p.size AS pizza_size,
  p.price AS unit_price,

  (od.quantity * p.price) AS revenue,
  pt.category AS pizza_category,
  pt.ingredients AS pizza_ingredients
FROM
  order_details od
LEFT JOIN orders o ON od.order_id = o.order_id
LEFT JOIN pizzas p ON od.pizza_id = p.pizza_id
LEFT JOIN pizza_types pt ON p.pizza_type_id = pt.pizza_type_id
"""

duckdb.sql(sql_create_master)

print("Preview Master Tabel (orders_full) dengan Kolom Baru:")

duckdb.sql("""
    SELECT order_id, order_date, order_time, month_name, day_name, hour, pizza_type, revenue
    FROM orders_full
    LIMIT 5
""").show()

Preview Master Tabel (orders_full) dengan Kolom Baru:
┌──────────┬────────────┬────────────┬────────────┬──────────┬───────┬───────────────────────────┬─────────┐
│ order_id │ order_date │ order_time │ month_name │ day_name │ hour  │        pizza_type         │ revenue │
│  int64   │    date    │    time    │  varchar   │ varchar  │ int64 │          varchar          │ double  │
├──────────┼────────────┼────────────┼────────────┼──────────┼───────┼───────────────────────────┼─────────┤
│        1 │ 2015-01-01 │ 11:38:36   │ January    │ Thursday │    11 │ The Hawaiian Pizza        │   13.25 │
│        2 │ 2015-01-01 │ 11:57:40   │ January    │ Thursday │    11 │ The Classic Deluxe Pizza  │    16.0 │
│        2 │ 2015-01-01 │ 11:57:40   │ January    │ Thursday │    11 │ The Five Cheese Pizza     │    18.5 │
│        2 │ 2015-01-01 │ 11:57:40   │ January    │ Thursday │    11 │ The Italian Supreme Pizza │   20.75 │
│        2 │ 2015-01-01 │ 11:57:40   │ January    │ Thursday │    11 │ The

In [52]:
sql_kpi = """
SELECT
  ROUND(SUM(revenue), 2) AS total_revenue,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS total_pizzas_sold,
  ROUND(SUM(revenue) / COUNT(DISTINCT order_id), 2) AS avg_order_value
FROM orders_full
"""

duckdb.sql(sql_kpi).show()

┌───────────────┬──────────────┬───────────────────┬─────────────────┐
│ total_revenue │ total_orders │ total_pizzas_sold │ avg_order_value │
│    double     │    int64     │      int128       │     double      │
├───────────────┼──────────────┼───────────────────┼─────────────────┤
│     817860.05 │        21350 │             49574 │           38.31 │
└───────────────┴──────────────┴───────────────────┴─────────────────┘



Top 10 by Category

In [56]:
sql_top10_quantity = """
SELECT
  pizza_type,
  SUM(quantity) AS total_quantity,
FROM orders_full
GROUP BY pizza_type
ORDER BY total_quantity DESC
LIMIT 10
"""

duckdb.sql(sql_top10_quantity).show()

┌──────────────────────────────┬────────────────┐
│          pizza_type          │ total_quantity │
│           varchar            │     int128     │
├──────────────────────────────┼────────────────┤
│ The Classic Deluxe Pizza     │           2453 │
│ The Barbecue Chicken Pizza   │           2432 │
│ The Hawaiian Pizza           │           2422 │
│ The Pepperoni Pizza          │           2418 │
│ The Thai Chicken Pizza       │           2371 │
│ The California Chicken Pizza │           2370 │
│ The Sicilian Pizza           │           1938 │
│ The Spicy Italian Pizza      │           1924 │
│ The Southwest Chicken Pizza  │           1917 │
│ The Big Meat Pizza           │           1914 │
├──────────────────────────────┴────────────────┤
│ 10 rows                             2 columns │
└───────────────────────────────────────────────┘



Top 10 by Revenue

In [57]:
sql_top10_revenue = """
SELECT
  pizza_type,
  ROUND(SUM(revenue), 2) AS total_revenue,
FROM orders_full
GROUP BY pizza_type
ORDER BY total_revenue DESC
LIMIT 10
"""

duckdb.sql(sql_top10_revenue).show()

┌──────────────────────────────┬───────────────┐
│          pizza_type          │ total_revenue │
│           varchar            │    double     │
├──────────────────────────────┼───────────────┤
│ The Thai Chicken Pizza       │      43434.25 │
│ The Barbecue Chicken Pizza   │       42768.0 │
│ The California Chicken Pizza │       41409.5 │
│ The Classic Deluxe Pizza     │       38180.5 │
│ The Spicy Italian Pizza      │      34831.25 │
│ The Southwest Chicken Pizza  │      34705.75 │
│ The Italian Supreme Pizza    │      33476.75 │
│ The Hawaiian Pizza           │      32273.25 │
│ The Four Cheese Pizza        │       32265.7 │
│ The Sicilian Pizza           │       30940.5 │
├──────────────────────────────┴───────────────┤
│ 10 rows                            2 columns │
└──────────────────────────────────────────────┘



Hourly Trend

In [58]:
sql_hourly = """
SELECT
    hour AS order_hour,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT order_id) AS total_orders
FROM orders_full
GROUP BY hour
ORDER BY hour;
"""
duckdb.sql(sql_hourly).show()

┌────────────┬───────────────┬──────────────┐
│ order_hour │ total_revenue │ total_orders │
│   int64    │    double     │    int64     │
├────────────┼───────────────┼──────────────┤
│          9 │          83.0 │            1 │
│         10 │        303.65 │            8 │
│         11 │       44935.8 │         1231 │
│         12 │      111877.9 │         2520 │
│         13 │      106065.7 │         2455 │
│         14 │       59201.4 │         1472 │
│         15 │       52992.3 │         1468 │
│         16 │       70055.4 │         1920 │
│         17 │      86237.45 │         2336 │
│         18 │      89296.85 │         2399 │
│         19 │       72628.9 │         2009 │
│         20 │       58215.4 │         1642 │
│         21 │       42029.8 │         1198 │
│         22 │      22815.15 │          663 │
│         23 │       1121.35 │           28 │
├────────────┴───────────────┴──────────────┤
│ 15 rows                         3 columns │
└─────────────────────────────────

Monthly Trend

In [43]:
sql_monthly = """
SELECT
    month_name AS order_month,
    ROUND(SUM(revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity
FROM orders_full
GROUP BY month_name
ORDER BY array_position([
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
], month_name);
"""

duckdb.sql(sql_monthly).show()

┌─────────────┬───────────────┬────────────────┐
│ order_month │ total_revenue │ total_quantity │
│   varchar   │    double     │     int128     │
├─────────────┼───────────────┼────────────────┤
│ January     │       69793.3 │           4232 │
│ February    │       65159.6 │           3961 │
│ March       │       70397.1 │           4261 │
│ April       │       68736.8 │           4151 │
│ May         │      71402.75 │           4328 │
│ June        │       68230.2 │           4107 │
│ July        │       72557.9 │           4392 │
│ August      │      68278.25 │           4168 │
│ September   │      64180.05 │           3890 │
│ October     │       64027.6 │           3883 │
│ November    │      70395.35 │           4266 │
│ December    │      64701.15 │           3935 │
├─────────────┴───────────────┴────────────────┤
│ 12 rows                            3 columns │
└──────────────────────────────────────────────┘



Daily Trend

In [45]:
sql_daily = """
SELECT
    day_name AS order_day,
    ROUND(SUM(revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity
FROM orders_full
GROUP BY day_name
ORDER BY array_position([
    'Sunday','Monday', 'Tuesday', 'Wednesday', 'Thursday',
    'Friday', 'Saturday'
], day_name);
"""

duckdb.sql(sql_daily).show()

┌───────────┬───────────────┬────────────────┐
│ order_day │ total_revenue │ total_quantity │
│  varchar  │    double     │     int128     │
├───────────┼───────────────┼────────────────┤
│ Sunday    │       99203.5 │           6035 │
│ Monday    │     107329.55 │           6485 │
│ Tuesday   │      114133.8 │           6895 │
│ Wednesday │      114408.4 │           6946 │
│ Thursday  │      123528.5 │           7478 │
│ Friday    │      136073.9 │           8242 │
│ Saturday  │      123182.4 │           7493 │
└───────────┴───────────────┴────────────────┘



In [47]:
sql_category = """
SELECT
    pizza_category,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(
        SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (),
        2
    ) AS revenue_percentage
FROM orders_full
GROUP BY pizza_category
ORDER BY total_revenue DESC;
"""

duckdb.sql(sql_category).show()

┌────────────────┬───────────────┬────────────────────┐
│ pizza_category │ total_revenue │ revenue_percentage │
│    varchar     │    double     │       double       │
├────────────────┼───────────────┼────────────────────┤
│ Classic        │      220053.1 │              26.91 │
│ Supreme        │      208197.0 │              25.46 │
│ Chicken        │      195919.5 │              23.96 │
│ Veggie         │     193690.45 │              23.68 │
└────────────────┴───────────────┴────────────────────┘



Performance by Size

In [61]:
sql_size = """
SELECT
    pizza_size,
    ROUND(SUM(revenue), 2) AS total_revenue,
    ROUND(
        SUM(revenue) * 100.0 / SUM(SUM(revenue)) OVER (),
        2
    ) AS revenue_percentage
FROM orders_full
GROUP BY pizza_size
ORDER BY total_revenue DESC;
"""

duckdb.sql(sql_size).show()

┌────────────┬───────────────┬────────────────────┐
│ pizza_size │ total_revenue │ revenue_percentage │
│  varchar   │    double     │       double       │
├────────────┼───────────────┼────────────────────┤
│ L          │      375318.7 │              45.89 │
│ M          │     249382.25 │              30.49 │
│ S          │      178076.5 │              21.77 │
│ XL         │       14076.0 │               1.72 │
│ XXL        │        1006.6 │               0.12 │
└────────────┴───────────────┴────────────────────┘



TOP 10 Worst Sales Pizza

In [51]:
top10_worst = """
SELECT
    pizza_type,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM orders_full
GROUP BY pizza_type
ORDER BY total_revenue ASC
LIMIT 10;
"""
duckdb.sql(top10_worst).show()

┌──────────────────────────────┬───────────────┐
│          pizza_type          │ total_revenue │
│           varchar            │    double     │
├──────────────────────────────┼───────────────┤
│ The Brie Carre Pizza         │       11588.5 │
│ The Green Garden Pizza       │      13955.75 │
│ The Spinach Supreme Pizza    │      15277.75 │
│ The Mediterranean Pizza      │       15360.5 │
│ The Spinach Pesto Pizza      │       15596.0 │
│ The Calabrese Pizza          │      15934.25 │
│ The Italian Vegetables Pizza │      16019.25 │
│ The Soppressata Pizza        │      16425.75 │
│ The Chicken Pesto Pizza      │      16701.75 │
│ The Chicken Alfredo Pizza    │      16900.25 │
├──────────────────────────────┴───────────────┤
│ 10 rows                            2 columns │
└──────────────────────────────────────────────┘

